# Homework #4: model building and tracking
### Xuanlin (Shirley) Liu - xl3572

**Instructions: For this assignment, we’d like you to use the F1 Datasets we have been using for the class to build any ML model of your choice and track the model for each run using MLflow. Select any of the F1 datasets in AWS S3 to build your model. You are allowed to join multiple datasets.**

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow.sklearn

import numpy as np
from sklearn.ensemble import RandomForestRegressor

import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tempfile

## 1. [20 pts] Build any model of your choice with tunable hyperparameters

In [0]:
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True).toPandas()


In [0]:
display(results)

In [0]:
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True).toPandas()

In [0]:
display(races)

In [0]:
df = results.merge(races[['raceId', 'year', 'round', 'circuitId']],on='raceId',how='left')

In [0]:
display(df)

In [0]:
cols = [
    'grid', 'driverId', 'constructorId', 'circuitId', 'year', 'round', 'positionOrder'
]

model_df = df[cols].copy()

# Drop missing rows
model_df = model_df.dropna()

# Ensure numeric
for c in model_df.columns:
    model_df[c] = pd.to_numeric(model_df[c], errors='coerce')

model_df = model_df.dropna()

In [0]:
X = model_df.drop('positionOrder', axis=1)
y = model_df['positionOrder']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# Example tunable hyperparameters
params = {
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": 42
}

# Build model
rf = RandomForestRegressor(**params)
rf.fit(X_train, y_train)

# Quick prediction check
predictions = rf.predict(X_test)
display(predictions[:10])

## 2. [20 pts] Create an experiment setup where - for each run - you log:

- the hyperparameters used in the model
- the model itself
- every possible metric from the model you chose
- at least two artifacts (plots, or csv files)


In [0]:
def log_rf_run(run_name, params, X_train, X_test, y_train, y_test):
    with mlflow.start_run(run_name=run_name) as run:
        # Train model
        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)
        predictions = rf.predict(X_test)

        # Log hyperparameters
        [mlflow.log_param(param, value) for param, value in params.items()]

        # Log model
        mlflow.sklearn.log_model(rf, "random_forest_model")

        # Metrics
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)

        # Log metrics
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        # Create temporary directory for artifacts
        temp_dir = tempfile.mkdtemp()

        # Artifact 1: predictions CSV
        pred_df = pd.DataFrame({
            "actual": y_test,
            "predicted": predictions
        })
        pred_path = os.path.join(temp_dir, "predictions.csv")
        pred_df.to_csv(pred_path, index=False)
        mlflow.log_artifact(pred_path, "artifacts")

        # Artifact 2: feature importance CSV
        importance_df = pd.DataFrame({
            "feature": X_train.columns,
            "importance": rf.feature_importances_
        }).sort_values("importance", ascending=False)
        importance_path = os.path.join(temp_dir, "feature_importance.csv")
        importance_df.to_csv(importance_path, index=False)
        mlflow.log_artifact(importance_path, "artifacts")

        # Artifact 3: actual vs predicted plot
        plt.figure(figsize=(8, 5))
        plt.scatter(y_test, predictions, alpha=0.5)
        plt.xlabel("Actual positionOrder")
        plt.ylabel("Predicted positionOrder")
        plt.title("Actual vs Predicted")
        plot_path = os.path.join(temp_dir, "actual_vs_predicted.png")
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_path, "plots")

        print("Run ID:", run.info.run_id)
        print("MSE:", mse)
        print("RMSE:", rmse)
        print("MAE:", mae)
        print("R2:", r2)

        return {
            "run_id": run.info.run_id,
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        }

## 3. [20 pts] Track your MLFlow experiment and run at least 10 experiments with different parameters each

In [0]:
param_grid = [
    {"n_estimators": 50,  "max_depth": 3,  "random_state": 42},
    {"n_estimators": 100, "max_depth": 3,  "random_state": 42},
    {"n_estimators": 200, "max_depth": 3,  "random_state": 42},
    {"n_estimators": 50,  "max_depth": 5,  "random_state": 42},
    {"n_estimators": 100, "max_depth": 5,  "random_state": 42},
    {"n_estimators": 200, "max_depth": 5,  "random_state": 42},
    {"n_estimators": 50,  "max_depth": 10, "random_state": 42},
    {"n_estimators": 100, "max_depth": 10, "random_state": 42},
    {"n_estimators": 200, "max_depth": 10, "random_state": 42},
    {"n_estimators": 300, "max_depth": 15, "random_state": 42}
]

In [0]:
results = []

for i, params in enumerate(param_grid, start=1):
    result = log_rf_run(
        run_name=f"F1_RF_Run_{i}",
        params=params,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test
    )

    results.append([
        f"Run {i}",
        result["run_id"],
        params["n_estimators"],
        params["max_depth"],
        result["mse"],
        result["rmse"],
        result["mae"],
        result["r2"]
    ])

In [0]:
results_df = pd.DataFrame(
    results,
    columns=["run_name", "run_id", "n_estimators", "max_depth", "mse", "rmse", "mae", "r2"]
)

display(results_df)

## 4. [20 pts] Select your best model run and explain why

In [0]:
best_run = results_df.sort_values("rmse", ascending=True).iloc[0]
display(pd.DataFrame([best_run]))

I selected Run 7 as the best model because it produced the lowest RMSE across all experiments. Since this is a regression problem, RMSE is an appropriate primary metric because it measures the typical size of the prediction error. A lower RMSE indicates that the model’s predictions are closer to the actual finishing positions.

Run 7 also performed well across the other regression metrics, with a relatively low MAE and the highest R^2 among the tested runs. This suggests that the model with n_estimators = 50 and max_depth = 10 provided the best balance of predictive accuracy and model complexity within the parameter combinations tested.

In [0]:
from sklearn.ensemble import RandomForestRegressor

best_params = {
    "n_estimators": 50,
    "max_depth": 10,
    "random_state": 42
}

best_model = RandomForestRegressor(**best_params)
best_model.fit(X_train, y_train)

best_predictions = best_model.predict(X_test)

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(y_test, best_predictions, alpha=0.5)
plt.xlabel("Actual positionOrder")
plt.ylabel("Predicted positionOrder")
plt.title("Run 7: Actual vs Predicted")

# ideal line
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()])

plt.show()

## 5. [20 pts] As part of your GitHub classroom submission include screenshots of

- your MLFlow Homepage
- your detailed run page

![](/Workspace/Users/xl3572@columbia.edu/take-home-exercise-3-xuanlin-liu/src/Screenshot 2026-04-13 at 7.12.55 PM.png)

![](/Workspace/Users/xl3572@columbia.edu/take-home-exercise-3-xuanlin-liu/src/Screenshot 2026-04-13 at 7.20.54 PM.png)

![](/Workspace/Users/xl3572@columbia.edu/take-home-exercise-3-xuanlin-liu/src/Screenshot 2026-04-13 at 7.20.30 PM.png)

![](/Workspace/Users/xl3572@columbia.edu/take-home-exercise-3-xuanlin-liu/src/Screenshot 2026-04-13 at 7.21.28 PM.png)

![](/Workspace/Users/xl3572@columbia.edu/take-home-exercise-3-xuanlin-liu/src/Screenshot 2026-04-13 at 7.21.54 PM.png)

![](/Workspace/Users/xl3572@columbia.edu/take-home-exercise-3-xuanlin-liu/src/Screenshot 2026-04-13 at 7.20.00 PM.png)

![](/Workspace/Users/xl3572@columbia.edu/take-home-exercise-3-xuanlin-liu/src/Screenshot 2026-04-13 at 7.19.23 PM.png)